# 05 — Transient analysis (onset-triggered averaging) + cold-start convergence

**No training.** Two mechanism-validation experiments:

**A. Onset-triggered averaging.** The gain-prior design rests on one specific claim
(`MODEL_GAIN_PRIOR.md` §2.2): the exported GR's 1024-sample (23 ms) trailing RMS
window *smears attack transients* — the fastest setting attacks in 1 ms ≈ 44 samples
— so the amplitude match systematically overshoots at note onsets, and the Δg head
exists to re-sharpen exactly there. Aggregate metrics can't see this (onsets are a
tiny fraction of samples). We detect onsets in the dry signal and average the
envelope error and Δg across hundreds of events, time-locked to the onset: if the
claim is right, (i) the amp-match error concentrates in the first ≈ 25 ms after
onsets, (ii) the model's error is smaller precisely there, (iii) mean Δg dips
(extra attenuation) right at the onset. The same protocol applied to the *detector*
shows the upstream attack-timing error.

**B. Cold-start convergence.** Both models are deployed as stateful stream
processors with a defined cold start; training masks 1.0 s of warmup for the
detector. Here we measure the *actual* convergence: prediction from a cold start at
a random song position vs the fully-warmed prediction at the same position, averaged
over many starts — the empirical justification (or correction) for the 1 s mask, and
the real-time-deployment spec ("how long until the output is trustworthy").

> ≈ 5–10 min on CPU (three full-song streams + many 3 s probes).

In [1]:
# -- 0. Setup ---------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import torch

from exp_common import (
    GR_DB_MAX, GR_DB_MIN, SR, amplitude_match, detect_onsets, ensure_eval_out,
    load_detector, load_gain_prior, load_pair, load_split, pairs_from_keys,
    params_for, short_env_db, stream_detector_gr, stream_gain_prior,
    stream_gain_prior_parts,
)

SELECTED_GAIN_PRIOR_RUN = "gain_prior_20260702_085618_diffssl_lstm32_gain_prior"
SELECTED_DETECTOR_RUN = "lstm_gr_20260702_174039_lstm_detector_gr"

MODEL, HP, RUN_DIR = load_gain_prior(SELECTED_GAIN_PRIOR_RUN)
DET, DET_HP, DET_DIR = load_detector(SELECTED_DETECTOR_RUN)
SPLIT = load_split(RUN_DIR)
VAL_PAIRS = pairs_from_keys(SPLIT.val_pair_keys)
TEST_PAIRS = pairs_from_keys(SPLIT.test_pair_keys)
OUT = ensure_eval_out()
HOP = int(DET.hop_size)

PAIRS = [VAL_PAIRS[2], VAL_PAIRS[7], TEST_PAIRS[0]]
ENV_WIN = 128          # 2.9 ms RMS - fast enough to resolve sub-label-window detail
PRE_MS, POST_MS = 60.0, 180.0

/Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 8,322-param GainPriorDiffSSLLSTM  (gain_prior_20260702_085618_diffssl_lstm32_gain_prior / best-059-730800.ckpt)


RuntimeError: Error(s) in loading state_dict for DetectorGRLSTM:
	Missing key(s) in state_dict: "film.weight", "film.bias", "glu.weight", "glu.bias". 
	size mismatch for lstm.weight_ih_l0: copying a param with shape torch.Size([128, 8]) from checkpoint, the shape in current model is torch.Size([128, 4]).

In [ ]:
# -- 1. Stream the pairs, detect onsets --------------------------------------
DATA = []
for song, setting in PAIRS:
    dry, wet, gr = load_pair(setting, song)
    p = params_for(setting)
    pred, delta_db, _ = stream_gain_prior_parts(MODEL, dry, gr, p)
    gr_det = stream_detector_gr(DET, dry, p, sample_len=dry.shape[-1])
    n = min(dry.shape[-1], pred.shape[-1], gr_det.shape[-1])
    onsets = detect_onsets(dry[..., :n])
    lo, hi = int(2.0 * SR), n - int(POST_MS / 1000 * SR) - 1
    onsets = onsets[(onsets > lo) & (onsets < hi)]
    DATA.append({"song": song, "setting": setting, "onsets": onsets,
                 "dry": dry[..., :n], "wet": wet[..., :n], "gr": gr[..., :n],
                 "pred": pred[..., :n], "delta": delta_db[..., :n],
                 "gr_det": gr_det[..., :n],
                 "amp": amplitude_match(dry[..., :n], gr[..., :n])})
    print(f"{song} / {setting}: {len(onsets)} onsets in {n/SR:.0f} s")

# sanity plot: detections on a 8 s window
d0 = DATA[0]
a, b = int(30 * SR), int(38 * SR)
t = np.arange(a, b) / SR
plt.figure(figsize=(13, 2.6))
plt.plot(t, d0["dry"][0, a:b].numpy(), lw=0.4, color="#444")
for o in d0["onsets"][(d0["onsets"] > a) & (d0["onsets"] < b)]:
    plt.axvline(o / SR, color="#d62728", lw=0.8, alpha=0.8)
plt.title(f"Onset detections - {d0['song']}"); plt.xlabel("time (s)")
plt.tight_layout(); plt.show()

In [ ]:
# -- 2. Onset-triggered averages: gain stage ----------------------------------
# For each onset, slice the SHORT-window envelope error (dB) of the amplitude
# match and of the model prediction, plus delta-g. Average time-locked.

pre, post = int(PRE_MS / 1000 * SR), int(POST_MS / 1000 * SR)
t_ms = (np.arange(-pre, post) / SR) * 1000


def triggered(sig_1d, onsets):
    return np.stack([sig_1d[o - pre:o + post] for o in onsets])


err_amp_ev, err_mod_ev, dg_ev = [], [], []
for d in DATA:
    env_wet = short_env_db(d["wet"], ENV_WIN)
    env_amp = short_env_db(d["amp"], ENV_WIN)
    env_mod = short_env_db(d["pred"], ENV_WIN)
    ok = env_wet > -60                                   # ignore silence frames
    ea = np.where(ok, np.abs(env_amp - env_wet), np.nan)
    em = np.where(ok, np.abs(env_mod - env_wet), np.nan)
    err_amp_ev.append(triggered(ea, d["onsets"]))
    err_mod_ev.append(triggered(em, d["onsets"]))
    dg_ev.append(triggered(d["delta"][0].numpy(), d["onsets"]))
err_amp_ev = np.concatenate(err_amp_ev)
err_mod_ev = np.concatenate(err_mod_ev)
dg_ev = np.concatenate(dg_ev)
n_ev = err_amp_ev.shape[0]
print(f"{n_ev} events pooled")


def mean_sem(ev):
    mu = np.nanmean(ev, axis=0)
    sem = np.nanstd(ev, axis=0) / np.sqrt(np.sum(~np.isnan(ev), axis=0))
    return mu, sem


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})
for ev, name, color in ((err_amp_ev, "amplitude match", "#1f77b4"),
                        (err_mod_ev, "gain-prior model", "#d62728")):
    mu, sem = mean_sem(ev)
    ax1.plot(t_ms, mu, color=color, lw=1.3, label=name)
    ax1.fill_between(t_ms, mu - 2 * sem, mu + 2 * sem, color=color, alpha=0.2)
ax1.axvline(0, color="k", lw=0.6); ax1.axvspan(0, 23.2, color="gray", alpha=0.12)
ax1.set_ylabel(f"|envelope error| (dB, {ENV_WIN/SR*1000:.1f} ms RMS)")
ax1.set_title(f"Onset-triggered envelope error ({n_ev} events; shaded = label RMS window)")
ax1.grid(alpha=0.3); ax1.legend()

mu, sem = mean_sem(dg_ev)
ax2.plot(t_ms, mu, color="#ff7f0e", lw=1.3)
ax2.fill_between(t_ms, mu - 2 * sem, mu + 2 * sem, color="#ff7f0e", alpha=0.2)
ax2.axvline(0, color="k", lw=0.6); ax2.axhline(0, color="k", lw=0.4, alpha=0.5)
ax2.axvspan(0, 23.2, color="gray", alpha=0.12)
ax2.set_xlabel("time relative to onset (ms)"); ax2.set_ylabel("mean delta-g (dB)")
ax2.set_title("Learned gain correction around onsets"); ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "05_onset_triggered_gain_stage.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 3. Onset-triggered averages: detector (stage 1 attack timing) ------------
gr_err_ev, gr_orc_ev, gr_det_ev = [], [], []
for d in DATA:
    orc = d["gr"][0].clamp(GR_DB_MIN, GR_DB_MAX).numpy()
    det = d["gr_det"][0].clamp(GR_DB_MIN, GR_DB_MAX).numpy()
    gr_err_ev.append(triggered(np.abs(det - orc), d["onsets"]))
    gr_orc_ev.append(triggered(orc, d["onsets"]))
    gr_det_ev.append(triggered(det, d["onsets"]))
gr_err_ev = np.concatenate(gr_err_ev)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
mu, sem = mean_sem(gr_err_ev)
ax1.plot(t_ms, mu, color="#9467bd", lw=1.3)
ax1.fill_between(t_ms, mu - 2 * sem, mu + 2 * sem, color="#9467bd", alpha=0.2)
ax1.axvline(0, color="k", lw=0.6); ax1.axvspan(0, 23.2, color="gray", alpha=0.12)
ax1.set_ylabel("|GR error| (dB)"); ax1.grid(alpha=0.3)
ax1.set_title("Detector GR error, onset-triggered (attack-timing cost of stage 1)")

ax2.plot(t_ms, np.concatenate(gr_orc_ev).mean(0), color="#1f77b4", lw=1.3, label="oracle GR")
ax2.plot(t_ms, np.concatenate(gr_det_ev).mean(0), color="#d62728", lw=1.3, label="predicted GR")
ax2.axvline(0, color="k", lw=0.6)
ax2.set_xlabel("time relative to onset (ms)"); ax2.set_ylabel("mean GR (dB)")
ax2.set_title("Mean GR trajectory around onsets"); ax2.grid(alpha=0.3); ax2.legend()
fig.tight_layout()
fig.savefig(OUT / "05_onset_triggered_detector.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 4. Cold-start convergence -------------------------------------------------
# Cold prediction from a random offset vs the fully-warmed full-song prediction
# at the same position. Detector: frame-rate |GR difference|. Gain stage:
# short-envelope dB difference of the outputs. K starts, mean over starts.

K, PROBE_SEC = 24, 3.0
rng = np.random.default_rng(42)
d0 = DATA[0]
n = d0["dry"].shape[-1]
probe = int(PROBE_SEC * SR) // HOP * HOP
offsets = (rng.integers(int(5 * SR), n - probe - 1, size=K) // HOP) * HOP

# detector
warm_fr = stream_detector_gr(DET, d0["dry"], params_for(d0["setting"]))  # frame rate
det_diff = []
for o in offsets:
    with torch.no_grad():
        cold = DET(d0["dry"][..., o:o + probe].unsqueeze(0),
                   params_for(d0["setting"])).squeeze().numpy()
    det_diff.append(np.abs(cold - warm_fr[0, o // HOP:o // HOP + len(cold)].numpy()))
det_diff = np.stack(det_diff)
t_fr = (np.arange(det_diff.shape[-1]) + 1) * HOP / SR

# gain stage (oracle GR input; state = main LSTM + tvcond)
gp_diff = []
for o in offsets:
    cold = stream_gain_prior(MODEL, d0["dry"][..., o:o + probe],
                             d0["gr"][..., o:o + probe], params_for(d0["setting"]))
    env_c = short_env_db(cold, 1024)
    env_w = short_env_db(d0["pred"][..., o:o + probe], 1024)
    gp_diff.append(np.abs(env_c - env_w))
gp_diff = np.stack(gp_diff)[:, 1024:]                  # drop the env-window edge
t_gp = (np.arange(gp_diff.shape[-1]) + 1024) / SR

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, diff, tt, name in ((ax1, det_diff, t_fr, "detector (|GR| diff, frame rate)"),
                           (ax2, gp_diff, t_gp, "gain stage (envelope dB diff)")):
    mu = diff.mean(0)
    ax.semilogy(tt, mu, lw=1.1, color="#d62728")
    ax.fill_between(tt, diff.min(0) + 1e-6, diff.max(0) + 1e-6, alpha=0.15, color="#d62728")
    ax.axvline(1.0, color="k", lw=0.8, ls="--", label="1 s training warmup mask")
    for thr_v in (0.1, 0.02):
        below = np.where(mu < thr_v)[0]
        tc = tt[below[0]] if len(below) else float("nan")
        print(f"{name}: converges below {thr_v} dB at t = {tc:.2f} s")
    ax.set_xlabel("time since cold start (s)"); ax.set_ylabel("|difference| (dB)")
    ax.set_title(name); ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8)
fig.suptitle(f"Cold-start convergence, {K} random starts - {d0['song']}")
fig.tight_layout()
fig.savefig(OUT / "05_cold_start_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

## Reading the results

- **Panel 2 (gain stage)**: the design claim is confirmed if the blue (amp-match)
  error rises sharply inside the shaded 23 ms window after onsets while the red
  (model) curve stays lower, *and* mean Δg swings negative in the same region
  (extra attenuation = re-sharpened attack). If Δg instead stays flat, the measured
  GR-MAE advantage comes from steady-state depth correction, not transient timing —
  a different (and citable) conclusion.
- **Panel 3 (detector)**: the onset-locked GR error is the perceptually weighted
  version of the 0.268 dB average — expect a multiple of it concentrated in the
  first ~30 ms. The mean-trajectory overlay shows whether the predictor is *late*
  (lag) or *shallow* (depth) at attacks, which discriminates between smoothing
  limits (frame rate, τ) and LSTM ballistics as the bottleneck.
- **Cold start**: the time at which the curves drop below 0.1 dB is the deployable
  warmup spec; if it is well under 1 s, the training mask is conservative (fine);
  if it exceeds 1 s, crop-head labels were noisier than assumed and the mask length
  is worth revisiting in the next training round (a *training* follow-up, noted here
  only as a measurement).